<a href="https://colab.research.google.com/github/francisco-mreina/olist-Predict-Insatisfaction-and-delay/blob/main/07_productivizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de clientes Olist (Brazilian E-Commerce)

## 10. Productivización

Tras haber construido y validado diversos modelos predictivos para la insatisfacción del cliente y el retraso de entregas, damos pie al paso final: invocar estos modelos desde fuera. Siguiendo el patrón de Modelo serializado con FastAPI del módulo, productivizamos aquí el caso de predicción de retraso en entrega: dado un pedido nuevo, ¿va a llegar tarde o a tiempo?

### 10.1. Diseño de la función/API de scoring (input: nuevo pedido → output: predicción)

En primer lugar, partimos del modelo xgb_retraso, ya entrenado y validado en el capítulo 6.2, que serializamos con pickle, junto con el mismo esquema de columnas del entrenamiento.

Esto nos permite con cada nuevo pedido en formato texto, poder recontruir las variables de la misma forma que las tiene el modelo, para que pueda "entenderlas".

Este orden se recupera directamente del modelo (`get_booster().feature_names`), sin depender de que las variables genéricas de entrenamiento sigan disponibles en memoria.

In [ ]:
import pickle

# Reutilizamos el modelo xgb_retraso ya entrenado en el capítulo 6.2
with open('modelo_retraso.pkl', 'wb') as file:
    pickle.dump({
        'modelo': xgb_retraso,
        'columnas': xgb_retraso.get_booster().feature_names,
        'predictores_originales': predictores_retraso_red,
    }, file)

print("Modelo de retraso serializado: modelo_retraso.pkl")

Modelo de retraso serializado: modelo_retraso.pkl


### 10.2. Servicio de scoring en el momento de creación del pedido

Con el modelo ya serializado, se define la API siguiendo exactamente el mismo patrón del módulo: una app de FastAPI, una clase Pydantic con las variables de entrada del pedido y un endpoint `/predict` que aplica el modelo y devuelve el resultado.

El servicio se lanza en un hilo separado, tal como se hace en el ejemplo del curso, para que `uvicorn` quede escuchando peticiones sin bloquear la ejecución del notebook. La idea que, en el instante en que la plataforma registra un pedido nuevo, el sistema de creación de pedidos llame a este endpoint para decidir si conviene marcarlo para seguimiento proactivo, en línea con las recomendaciones de los capítulos 7.2, 8.1 y 9.3.

In [ ]:
# Creamos la puerta de entrada (API), la llamada al modelo para predecir retraso

app = FastAPI(
    title="API de Scoring de Retraso — TFM Olist",
    description="Predice si un pedido nuevo tendrá retraso en la entrega.",
    version="1.0.0",
    contact={
        "name": "Francisco Morales Reina",
        "email": "fran.moreina@gmail.com",
    },
)

# Cargar el modelo serializado
with open('modelo_retraso.pkl', 'rb') as file:
    artefacto_retraso = pickle.load(file)

REGIONES_VALIDAS = ['Centro-Oeste', 'Nordeste', 'Norte', 'Sudeste', 'Sur']

# Clase para la entrada de datos: las variables en crudo del pedido nuevo
class PedidoFeatures(BaseModel):
    customer_region: Literal['Centro-Oeste', 'Nordeste', 'Norte', 'Sudeste', 'Sur']
    n_items: int
    n_pagos: int
    max_installments: int
    peso_total_g: float
    order_purchase_hour: int
    ticket_medio_item: float
    pct_flete: float


def _preparar_fila(pedido: PedidoFeatures) -> pd.DataFrame:
    fila = pd.DataFrame([pedido.model_dump()])
    region = fila.pop('customer_region').iloc[0]
    for r in REGIONES_VALIDAS:
        col = f'customer_region_{r}'
        if col in artefacto_retraso['columnas']:
            fila[col] = 1 if region == r else 0
    return fila.reindex(columns=artefacto_retraso['columnas'], fill_value=0)


# Endpoint para hacer predicciones
@app.post("/predict")
def predict(pedido: PedidoFeatures):
    X = _preparar_fila(pedido)
    prob = float(artefacto_retraso['modelo'].predict_proba(X)[:, 1][0])
    prediccion = int(prob >= 0.5)
    etiqueta = ['A tiempo', 'Con retraso'][prediccion]
    return {"prediccion": etiqueta, "probabilidad_retraso": round(prob, 4)}



def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Correr en un hilo separado para evitar bloquear el notebook
thread = threading.Thread(target=run)
thread.start()

### 10.3. Demo (script, notebook interactivo)

Con el servicio ya levantado en el hilo anterior, se lanza una petición de prueba desde otra celda del notebook, exactamente como en el módulo: se construye el JSON de un pedido nuevo y se envía por `POST` con la librería `requests`.

In [ ]:
# Lanzo una petición al modelo de retraso

url = 'http://localhost:8000/predict'

data = {
    "customer_region": "Nordeste",
    "n_items": 3,
    "n_pagos": 1,
    "max_installments": 6,
    "peso_total_g": 8500,
    "order_purchase_hour": 22,
    "ticket_medio_item": 310.0,
    "pct_flete": 0.28
}

response = requests.post(url, json=data)
print(response.json())

INFO:     Started server process [825]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:57020 - "POST /predict HTTP/1.1" 200 OK
{'prediccion': 'A tiempo', 'probabilidad_retraso': 0.1004}


Como demo interactiva adicional dentro del propio notebook, se ofrece también un pequeño formulario con `ipywidgets` que permite variar los datos del pedido y llamar al mismo endpoint `/predict` sin tener que editar el JSON a mano cada vez.

In [ ]:
w_region = widgets.Dropdown(options=REGIONES_VALIDAS, description='Región:')
w_flete = widgets.FloatSlider(value=0.15, min=0, max=1, step=0.01, description='% flete:')
w_ticket = widgets.FloatText(value=150.0, description='Ticket medio:')
w_hora = widgets.IntSlider(value=14, min=0, max=23, description='Hora compra:')
w_peso = widgets.FloatText(value=2000.0, description='Peso (g):')
boton = widgets.Button(description='Calcular riesgo de retraso')
salida = widgets.Output()


def calcular_riesgo(b):
    with salida:
        salida.clear_output()
        pedido = {
            'customer_region': w_region.value,
            'n_items': 1,
            'n_pagos': 1,
            'max_installments': 1,
            'peso_total_g': w_peso.value,
            'order_purchase_hour': w_hora.value,
            'ticket_medio_item': w_ticket.value,
            'pct_flete': w_flete.value,
        }
        resultado = requests.post(url, json=pedido).json()
        print(resultado)


boton.on_click(calcular_riesgo)
display(w_region, w_flete, w_ticket, w_hora, w_peso, boton, salida)

Dropdown(description='Región:', options=('Centro-Oeste', 'Nordeste', 'Norte', 'Sudeste', 'Sur'), value='Centro…

FloatSlider(value=0.15, description='% flete:', max=1.0, step=0.01)

FloatText(value=150.0, description='Ticket medio:')

IntSlider(value=14, description='Hora compra:', max=23)

FloatText(value=2000.0, description='Peso (g):')

Button(description='Calcular riesgo de retraso', style=ButtonStyle())

Output()